In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("data/healthcare_dataset.csv")
df = data.copy()

# Clean the 'Name' column
df['Name'] = df['Name'].str.strip()
df['Name'] = df['Name'].str.title()
df['Name'] = df['Name'].str.replace(r"^(dr|mr|mrs|ms|miss)\.?\s+", "", case=False, regex=True)
df['Name'] = df['Name'].str.replace(r"\s+(md|dds|dvm|phd|jr\.?|ii|iii|iv)$", "", case=False, regex=True)

#print(df.head()) 
#looks okay so far

#print(df.duplicated().sum())
#print("Full-row duplicates in data:", data.duplicated().sum())
#print("Full-row duplicates after cleaning names:", df.duplicated().sum())
#print("Repeated names (not necessarily duplicates):", df["Name"].duplicated().sum())

for col in df.columns:
    others = df.columns.drop(col)
    count = df.duplicated(subset=others).sum()
    if count > 534:
        print(f"Ignoring {col}: {count} duplicates")

#theres a tricky age duplicate issue, we need to figure out how to handle that.


Ignoring Age: 5500 duplicates


In [2]:
#gettiung rid of exact duplicates, we need to figure out other duplicates next
df = df.drop_duplicates()
print("Rows after dropping exact duplicates:", len(df))

Rows after dropping exact duplicates: 54966


In [3]:
#weird situation where we have duplicates in all columns except age, we need to figure out how to handle that. which age is right?

key = df.columns.drop("Age").tolist()

almost_dupes = df[df.duplicated(subset=key, keep=False) & ~df.duplicated(keep=False)]
example = almost_dupes["Name"].iloc[0]

df[df["Name"] == example].sort_values("Date of Admission")

#id imagine medical facilities would use birthdate rather than age, using date of admission and birthdate to calculate age would be more accurate.
#ive decided to keep the first instance of the duplicate and drop the rest, this way we have some older and some newer ages without having duplicates.

key = df.columns.drop("Age").tolist()
grouped = df.groupby(key)["Age"]

print("Keep first:", grouped.first().mean().round(2))
print("Keep younger:", grouped.min().mean().round(2))
print("Keep older:", grouped.max().mean().round(2))

#minimal difference regardless of which age we keep, so we'll just keep the first instance of the duplicate and drop the rest.

Keep first: 51.58
Keep younger: 51.42
Keep older: 51.69


In [4]:
key = df.columns.drop("Age").tolist()

df["Age duplication"] = df.groupby(key)["Age"].transform("nunique") > 1
df = df.drop_duplicates(subset=key, keep="first")

print("Rows remaining:", len(df))
print("Records with conflicting ages:", df["Age duplication"].sum())
print("Duplicates left:", df.duplicated(subset=key).sum())

Rows remaining: 50000
Records with conflicting ages: 4966
Duplicates left: 0


In [5]:
#fixing the date of admission column, we want to make sure all dates are in the same format and that they are valid dates.

for col in ["Date of Admission", "Discharge Date"]:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")

print(df[["Date of Admission", "Discharge Date"]].dtypes)
print("Missing admission dates:", df["Date of Admission"].isna().sum())
print("Missing discharge dates:", df["Discharge Date"].isna().sum())
print(df[["Date of Admission", "Discharge Date"]].sample(5))

Date of Admission    datetime64[us]
Discharge Date       datetime64[us]
dtype: object
Missing admission dates: 0
Missing discharge dates: 0
      Date of Admission Discharge Date
22634        2023-01-03     2023-01-11
37535        2019-08-12     2019-09-03
31642        2024-03-28     2024-04-27
13547        2023-09-05     2023-10-04
22179        2023-05-05     2023-05-27


In [6]:
#what should we do with the negative billing amounts? 
#personally im curious as to how they affect the overall billing amounts, ive decided to create different senarios that make sense to me and see how they affect the overall billing amounts.

if "Original Billing Amount" not in df.columns:
    df["Original Billing Amount"] = df["Billing Amount"].round(2)

neg_mask = df["Original Billing Amount"] < 0
negative_bills = df[neg_mask].copy()
negative_bills.to_csv("data/negative_bills.csv", index=False)

print("Negative bills:", len(negative_bills))
print("Total negative amount:", round(negative_bills["Original Billing Amount"].sum(), 2))
print("Total positive amount:", round(df.loc[~neg_mask, "Original Billing Amount"].sum(), 2))

df["Billing Amount"] = df["Original Billing Amount"].where(~neg_mask)

Negative bills: 96
Total negative amount: -47650.4
Total positive amount: 1277832227.07


In [7]:
print("Blank bills:", df["Billing Amount"].isna().sum())
print("Smallest bill:", df["Billing Amount"].min())
print("Most decimal places:", df["Billing Amount"].dropna().astype(str).str.split(".").str[1].str.len().max())

Blank bills: 96
Smallest bill: 9.24
Most decimal places: 2


In [8]:
#cleaning other columns with names, such as hospital name, doctor name, and insurance provider. we want to make sure all names are in the same format and that they are valid names.
hospital = df["Hospital"]

h = df["Hospital"]

print("Unique hospitals:", h.nunique())
print("Ending with a comma:", h.str.endswith(",").sum())
print("Starting with 'and':", h.str.lower().str.startswith("and ").sum())
print("Extra spaces at the edges:", (h != h.str.strip()).sum())
print("Double spaces inside:", h.str.contains(r"\s{2,}", regex=True).sum())
print("Leading punctuation:", h.str.match(r"^[^\w]").sum())
print("Unusual characters:", h.str.contains(r"[^\w\s,.\-&']", regex=True).sum())

hospital[hospital.str.endswith(",") | hospital.str.lower().str.startswith("and ")].sample(10, random_state=1)

Unique hospitals: 39876
Ending with a comma: 4299
Starting with 'and': 5158
Extra spaces at the edges: 0
Double spaces inside: 0
Leading punctuation: 0
Unusual characters: 0


18377    and Hubbard Juarez Carter,
2726     and Patterson, Lam Coleman
668                   and Sons Ford
19923       Carroll and Hall Meyer,
10734          Chase Key and Clark,
11858       Cook and Curtis Morris,
14716       Jones and Taylor Baker,
24284     Harrison and Torres Howe,
30870       and Salinas, Wang Lucas
39935    and White, Fisher Anderson
Name: Hospital, dtype: str

In [9]:
df["Hospital"] = (
    df["Hospital"].str.strip()
    .str.replace(r"^and\s+", "", case=False, regex=True)
    .str.rstrip(",")
    .str.strip()
)

print("Unique hospitals after cleaning:", df["Hospital"].nunique())
#Better safe than sorry, there wasnt much issue with the hospital names, but we want to make sure all names are in the same format and that they are valid names.

Unique hospitals after cleaning: 39876


In [10]:
#doctor and hospital can have non title case and periods because of Dr. and LLC things like that
for col in ["Name", "Doctor", "Hospital"]:
    s = df[col]
    print(f"{col}:")
    print("  Edge spaces:", (s != s.str.strip()).sum())
    print("  Double spaces:", s.str.contains(r"\s{2,}", regex=True).sum())
    print("  Not title case:", (s != s.str.title()).sum())
    print("  Contains a period:", s.str.contains(r"\.", regex=True).sum())

Name:
  Edge spaces: 0
  Double spaces: 0
  Not title case: 0
  Contains a period: 0
Doctor:
  Edge spaces: 0
  Double spaces: 0
  Not title case: 1104
  Contains a period: 1047
Hospital:
  Edge spaces: 0
  Double spaces: 0
  Not title case: 19916
  Contains a period: 0


In [11]:
#saving cleaned file to csv, we want to make sure all names are in the same format and that they are valid names.
df.to_csv("data/healthcare_clean.csv", index=False)
print("Saved:", df.shape)
df = pd.read_csv("data/healthcare_clean.csv", parse_dates=["Date of Admission", "Discharge Date"])

Saved: (50000, 17)
